# BuzzSpot RF-DETR Large 1536 — true resume from 12 to 18 total epochs

This notebook continues the completed **1344 → 1536 warm-start adaptation run**
for exactly six additional epochs.

It is a **true PyTorch Lightning resume**:

1. finds the latest full checkpoint from the existing 12-epoch run directory;
2. verifies that model, optimizer, scheduler, callback, epoch, and global-step
   states are present;
3. resumes from the next epoch;
4. raises the trainer's total epoch limit from 12 to 18;
5. keeps the configured base learning rates at `5e-5`;
6. sets `warmup_epochs=0.0`, so no new warmup phase is constructed;
7. keeps the original 12-epoch selected EMA checkpoint backed up before training.

The notebook refuses to silently fall back to a fresh warm start. If a valid full
resume checkpoint cannot be found, it stops before training.

All other training settings remain unchanged:

- resolution: 1536
- batch size: 2
- gradient accumulation: 8
- effective batch size: 16
- EMA enabled
- same train+valid oversampled dataset
- same augmentation and class multipliers
- evaluation and checkpointing every epoch


## 1. Install
- Uses a pinned RF-DETR/Pillow environment with a one-time runtime restart and early import smoke test.


In [17]:
# Reproducible RF-DETR environment setup for Google Colab.
# This cell intentionally restarts the runtime ONCE after installation so Pillow
# and all compiled/imported modules are loaded from one consistent installation.

import os
import sys
import subprocess
from pathlib import Path

SETUP_MARKER = Path("/content/.buzzspot_rfdetr_env_ready_v3")

if not SETUP_MARKER.exists():
    print("Installing pinned RF-DETR environment...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
        "rfdetr[train,loggers]==1.8.3",
        "pycocotools",
        "tqdm",
        "supervision>=0.29.0",
    ])

    # Repair/prevent mixed Pillow files in long-lived Colab runtimes.
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
        "--force-reinstall", "Pillow==12.3.0",
    ])

    SETUP_MARKER.write_text("ready")
    print("Environment installed. Restarting the Colab runtime once...")
    print("After reconnecting, run the notebook again from the top.")
    os.kill(os.getpid(), 9)
else:
    print("Pinned environment already installed; continuing without another restart.")


Pinned environment already installed; continuing without another restart.


## 1.1 Environment import smoke test

This runs before any dataset extraction so dependency problems fail immediately.


In [18]:
import PIL
import rfdetr
import supervision as sv
from PIL import Image, ImageText
from rfdetr import RFDETRLarge

print("Pillow:", PIL.__version__)
print("RF-DETR import: OK")
print("supervision:", sv.__version__)
print("RFDETRLarge class:", RFDETRLarge)


Pillow: 12.3.0
RF-DETR import: OK
supervision: 0.29.1
RFDETRLarge class: <class 'rfdetr.variants.RFDETRLarge'>


## 2. Mount Drive and locate dataset files

In [19]:
from google.colab import drive
drive.mount("/content/drive")

import glob
from pathlib import Path

zip_hits = glob.glob("/content/drive/MyDrive/**/BuzzSet_challenge.zip", recursive=True)
assert zip_hits, "BuzzSet_challenge.zip not found under MyDrive"
ZIP_PATH = zip_hits[0]

tar_hits = glob.glob("/content/drive/MyDrive/**/19557529600-BuzzSet_challenge_testphase.tar", recursive=True)
if not tar_hits:
    tar_hits = glob.glob("/content/drive/MyDrive/**/*BuzzSet_challenge_testphase*.tar*", recursive=True)
assert tar_hits, "test-phase tar file not found under MyDrive"
TAR_PATH = tar_hits[0]

print("old train/valid zip:", ZIP_PATH)
print("new test-phase tar:", TAR_PATH)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
old train/valid zip: /content/drive/MyDrive/Colab Notebooks/CVPPA ECCV 26 BuzzSpot Challenge/BuzzSet_challenge.zip
new test-phase tar: /content/drive/MyDrive/Colab Notebooks/CVPPA ECCV 26 BuzzSpot Challenge/19557529600-BuzzSet_challenge_testphase.tar


## 3. Config

In [20]:
from pathlib import Path
import shutil

# Main switch
ENABLE_TRAINING = False  # 1344 EMA -> 1536 warm-start adaptation.

# Reliability controls
AUTO_RESUME = True              # Locate the newest full checkpoint in the existing run directory.
REQUIRE_TRUE_RESUME = True      # Never silently start another warm-start run.
ENABLE_MEMORY_PREFLIGHT = False # A verified full resume does not need a memory preflight.
PREFLIGHT_TRAIN_IMAGES = 4      # Highest-annotation train images; enough for two batch-2 steps.
PREFLIGHT_VALID_IMAGES = 2

# Inference/eval switches
ENABLE_REGULAR_VALIDATION = True       # leaked sanity only
ENABLE_REGULAR_INFERENCE = True
ENABLE_SAHI_INFERENCE = False

MODEL_SIZE = "large"
SOURCE_MODEL_TAG = "rfdetr_large_trainval_os_1344_32ep"
MODEL_TAG = "rfdetr_large_trainval_os_1344to1536_18ep"
RUN_NAME = "buzz_rfdetr_large_trainval_os_1344to1536_12ep"  # Existing run directory; do not rename.

# Warm-start adaptation setup
SOURCE_RESOLUTION = 1344
RESOLUTION = 1536
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8    # effective batch = 16; if OOM use 1 and 16
EPOCHS = 18
LR = 5e-5
LR_ENCODER = 5e-5
WARMUP_EPOCHS = 0.0
LR_SCHEDULER = "step"  # Same scheduler family used by the completed run.
LR_DROP = 100          # No scheduled drop during an 18-epoch run.
SKIP_BEST_EPOCHS = 1    # Ignore epoch 0 when selecting the best checkpoint.
USE_EMA = True
EARLY_STOPPING = False  # fixed run; valid is leaked because train+val are combined
GRADIENT_CHECKPOINTING = True
CHECKPOINT_INTERVAL = 1
EVAL_INTERVAL = 1
DEVICE = "cuda"
SEED = 0

# Rare-class oversampling. For multi-class images, use the max multiplier.
# 1 bee, 2 bumblebee, 3 hoverfly, 4 moth
CATEGORY_MULTIPLIERS = {
    1: 1,
    2: 4,
    3: 2,
    4: 5,
}

# Regular RF-DETR inference. Keep low for AP-style ranked detections.
REGULAR_CONF = 0.01

# Optional manual SAHI inference settings. Disabled by default.
SAHI_CONF = 0.05
SAHI_SLICE_SIZE = 700
SAHI_OVERLAP = 0.20
SAHI_MATCH_THRESHOLD = 0.50
SAHI_MAX_DET_PER_IMAGE = 300

PROJECT_DIR = Path("/content/drive/MyDrive/BuzzSpot")
LOCAL_DATA_DIR = Path("/content/buzz_rfdetr_trainval_os")
DRIVE_RUNS_DIR = PROJECT_DIR / "runs_rfdetr"
WEIGHTS_DIR = PROJECT_DIR / "weights"
SUBMISSIONS_DIR = PROJECT_DIR / "submissions"

for d in [DRIVE_RUNS_DIR, WEIGHTS_DIR, SUBMISSIONS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Source: the exact selected 1344 EMA checkpoint that produced the 0.4049 submission.
SOURCE_1344_WEIGHTS = WEIGHTS_DIR / f"{SOURCE_MODEL_TAG}_selected_for_inference.pth"
LOCAL_SOURCE_1344_WEIGHTS = Path("/content/rfdetr_1344_selected_for_warmstart.pth")

# Continue inside the existing 12-epoch run directory so its full Lightning
# checkpoint and callback state remain available.
DRIVE_RUN_DIR = DRIVE_RUNS_DIR / RUN_NAME

PREVIOUS_SELECTED_WEIGHTS = (
    WEIGHTS_DIR
    / "rfdetr_large_trainval_os_1344to1536_12ep_selected_for_inference.pth"
)
PRE_EXTENSION_SELECTED_BACKUP = (
    WEIGHTS_DIR
    / "rfdetr_large_trainval_os_1344to1536_12ep_selected_before_18ep_resume.pth"
)
PRE_EXTENSION_RUN_BEST_BACKUP = (
    DRIVE_RUN_DIR / "checkpoint_best_ema_before_18ep_resume.pth"
)

DRIVE_SELECTED_WEIGHTS = WEIGHTS_DIR / f"{MODEL_TAG}_selected_for_inference.pth"
LOCAL_WEIGHTS = Path("/content/rfdetr_1344to1536_18ep_selected_for_inference.pth")

# Full Lightning checkpoints are discovered dynamically. In RF-DETR 1.8.3,
# interval checkpoints are normally .ckpt files, not checkpoint.pth.
RESUME_CHECKPOINT = None
PREFLIGHT_DATA_DIR = Path("/content/buzz_rfdetr_warmstart_memory_preflight")
PREFLIGHT_OUTPUT_DIR = Path("/content/rfdetr_warmstart_memory_preflight_output")

REG_TAG = f"regular_conf{int(REGULAR_CONF * 1000):03d}"
SAHI_TAG = f"sahi_conf{int(SAHI_CONF * 1000):03d}_slice{SAHI_SLICE_SIZE}_ov{int(SAHI_OVERLAP*100):02d}"

LOCAL_REG_PRED_PATH = Path(f"/content/predictions_{MODEL_TAG}_{REG_TAG}.json")
LOCAL_REG_ZIP_OUT = Path(f"/content/submission_{MODEL_TAG}_{REG_TAG}.zip")
DRIVE_REG_PRED_PATH = SUBMISSIONS_DIR / LOCAL_REG_PRED_PATH.name
DRIVE_REG_ZIP_OUT = SUBMISSIONS_DIR / LOCAL_REG_ZIP_OUT.name

LOCAL_SAHI_PRED_PATH = Path(f"/content/predictions_{MODEL_TAG}_{SAHI_TAG}.json")
LOCAL_SAHI_ZIP_OUT = Path(f"/content/submission_{MODEL_TAG}_{SAHI_TAG}.zip")
DRIVE_SAHI_PRED_PATH = SUBMISSIONS_DIR / LOCAL_SAHI_PRED_PATH.name
DRIVE_SAHI_ZIP_OUT = SUBMISSIONS_DIR / LOCAL_SAHI_ZIP_OUT.name

CLASS_NAMES = ["bee", "bumblebee", "hoverfly", "moth"]
CATEGORIES = [{"id": i + 1, "name": name, "supercategory": "pollinator"} for i, name in enumerate(CLASS_NAMES)]

print("ENABLE_TRAINING:", ENABLE_TRAINING)
print("SOURCE CHECKPOINT:", SOURCE_1344_WEIGHTS)
print("MODEL_TAG:", MODEL_TAG)
print("SOURCE_RESOLUTION:", SOURCE_RESOLUTION)
print("TARGET RESOLUTION:", RESOLUTION)
print("BATCH_SIZE:", BATCH_SIZE)
print("GRAD_ACCUM_STEPS:", GRAD_ACCUM_STEPS)
print("effective batch:", BATCH_SIZE * GRAD_ACCUM_STEPS)
print("EPOCHS:", EPOCHS)
print("LR / LR_ENCODER:", LR, LR_ENCODER)
print("SKIP_BEST_EPOCHS:", SKIP_BEST_EPOCHS)
print("CATEGORY_MULTIPLIERS:", CATEGORY_MULTIPLIERS)
print("AUTO_RESUME:", AUTO_RESUME)
print("REQUIRE_TRUE_RESUME:", REQUIRE_TRUE_RESUME)
print("ENABLE_MEMORY_PREFLIGHT:", ENABLE_MEMORY_PREFLIGHT)
print("RESUME SEARCH DIRECTORY:", DRIVE_RUN_DIR)
print("DRIVE_RUN_DIR:", DRIVE_RUN_DIR)
print("REGULAR ZIP:", DRIVE_REG_ZIP_OUT)


ENABLE_TRAINING: False
SOURCE CHECKPOINT: /content/drive/MyDrive/BuzzSpot/weights/rfdetr_large_trainval_os_1344_32ep_selected_for_inference.pth
MODEL_TAG: rfdetr_large_trainval_os_1344to1536_18ep
SOURCE_RESOLUTION: 1344
TARGET RESOLUTION: 1536
BATCH_SIZE: 2
GRAD_ACCUM_STEPS: 8
effective batch: 16
EPOCHS: 18
LR / LR_ENCODER: 5e-05 5e-05
SKIP_BEST_EPOCHS: 1
CATEGORY_MULTIPLIERS: {1: 1, 2: 4, 3: 2, 4: 5}
AUTO_RESUME: True
REQUIRE_TRUE_RESUME: True
ENABLE_MEMORY_PREFLIGHT: False
RESUME SEARCH DIRECTORY: /content/drive/MyDrive/BuzzSpot/runs_rfdetr/buzz_rfdetr_large_trainval_os_1344to1536_12ep
DRIVE_RUN_DIR: /content/drive/MyDrive/BuzzSpot/runs_rfdetr/buzz_rfdetr_large_trainval_os_1344to1536_12ep
REGULAR ZIP: /content/drive/MyDrive/BuzzSpot/submissions/submission_rfdetr_large_trainval_os_1344to1536_18ep_regular_conf010.zip


In [21]:
# Hard guards: fail before training if the warm-start configuration drifted.
assert MODEL_SIZE == "large"
assert SOURCE_RESOLUTION == 1344
assert RESOLUTION == 1536
assert EPOCHS == 18
assert LR == 5e-5
assert LR_ENCODER == 5e-5
assert WARMUP_EPOCHS == 0.0
assert LR_SCHEDULER == "step"
assert LR_DROP == 100
assert SKIP_BEST_EPOCHS == 1
assert BATCH_SIZE * GRAD_ACCUM_STEPS == 16
assert CATEGORY_MULTIPLIERS == {1: 1, 2: 4, 3: 2, 4: 5}
assert REGULAR_CONF == 0.01
assert AUTO_RESUME is True
assert ENABLE_MEMORY_PREFLIGHT is False
assert REQUIRE_TRUE_RESUME is True
assert PREFLIGHT_TRAIN_IMAGES >= BATCH_SIZE * 2
assert RESOLUTION % 32 == 0
assert SOURCE_1344_WEIGHTS != DRIVE_SELECTED_WEIGHTS
assert DRIVE_RUN_DIR.name == RUN_NAME
assert RUN_NAME.endswith("_12ep")
assert MODEL_TAG.endswith("_18ep")
assert PREVIOUS_SELECTED_WEIGHTS != DRIVE_SELECTED_WEIGHTS
print("1536 true-resume configuration guard passed.")


1536 true-resume configuration guard passed.


## 4. Build RF-DETR COCO dataset

In [22]:
import json, zipfile, tarfile, shutil, random
from pathlib import Path
from collections import defaultdict, Counter
from tqdm.auto import tqdm

random.seed(SEED)

if LOCAL_DATA_DIR.exists():
    shutil.rmtree(LOCAL_DATA_DIR)

for split in ["train", "valid", "test"]:
    (LOCAL_DATA_DIR / split).mkdir(parents=True, exist_ok=True)

zf = zipfile.ZipFile(ZIP_PATH)
tf = tarfile.open(TAR_PATH, "r:*")

zip_names = set(zf.namelist())
tar_members = {m.name: m for m in tf.getmembers() if m.isfile()}
tar_names = set(tar_members.keys())

def find_zip(rel):
    rel = rel.lstrip("/")
    for cand in (rel, f"BuzzSet_challenge/{rel}"):
        if cand in zip_names:
            return cand
    suffix = "/" + rel
    hits = [n for n in zip_names if n.endswith(suffix)]
    return hits[0] if hits else None

def find_tar(rel):
    rel = rel.lstrip("/")
    for cand in (rel, f"BuzzSet_challenge/{rel}", f"BuzzSet_challenge_testphase/{rel}"):
        if cand in tar_names:
            return cand
    suffix = "/" + rel
    hits = [n for n in tar_names if n.endswith(suffix)]
    return hits[0] if hits else None

def load_tar_json(fname):
    p = find_tar(f"annotations/{fname}") or find_tar(fname)
    assert p is not None, f"{fname} not found in tar"
    with tf.extractfile(tar_members[p]) as f:
        obj = json.load(f)
    print("loaded current annotation:", fname, "from", p)
    return obj

def flat_name(file_name):
    return file_name.replace("/", "__")

def clip_bbox_xywh(bbox, W, H):
    x, y, w, h = map(float, bbox)
    x = max(0.0, min(x, W - 1))
    y = max(0.0, min(y, H - 1))
    w = max(0.0, min(w, W - x))
    h = max(0.0, min(h, H - y))
    if w < 1 or h < 1:
        return None
    return [round(x, 2), round(y, 2), round(w, 2), round(h, 2)]

def anns_by_image(coco):
    by_img = defaultdict(list)
    for ann in coco.get("annotations", []):
        by_img[int(ann["image_id"])].append(ann)
    return by_img

def get_annotated_records(coco, zip_img_dir, source_split_name):
    by_img = anns_by_image(coco)
    annotated_ids = set(by_img.keys())
    records = []

    skipped_unannotated = 0
    class_counts = Counter()

    for im in coco["images"]:
        iid = int(im["id"])
        if iid not in annotated_ids:
            skipped_unannotated += 1
            continue

        W, H = int(im["width"]), int(im["height"])
        src = find_zip(f"{zip_img_dir}/{im['file_name']}") or find_zip(im["file_name"])
        assert src is not None, f"missing image in old zip: {zip_img_dir}/{im['file_name']}"

        anns = []
        for ann in by_img[iid]:
            bbox = clip_bbox_xywh(ann["bbox"], W, H)
            if bbox is None:
                continue
            cat = int(ann["category_id"])
            assert cat in {1, 2, 3, 4}, f"bad category_id: {cat}"
            anns.append({
                "category_id": cat,
                "bbox": bbox,
                "iscrowd": int(ann.get("iscrowd", 0)),
            })
            class_counts[cat] += 1

        if anns:
            records.append({
                "source_split": source_split_name,
                "orig_id": iid,
                "file_name": im["file_name"],
                "width": W,
                "height": H,
                "zip_member": src,
                "anns": anns,
            })

    print()
    print(f"{source_split_name} COCO image records:", len(coco["images"]))
    print(f"{source_split_name} annotated image ids used:", len(records))
    print(f"{source_split_name} skipped unannotated/context image records:", skipped_unannotated)
    print(f"{source_split_name} boxes:", sum(len(r["anns"]) for r in records))
    print(f"{source_split_name} class counts:", dict(class_counts))
    print(f"{source_split_name} named counts:", {CLASS_NAMES[k-1]: class_counts[k] for k in range(1, 5)})
    return records

def copy_zip_member(member, dst):
    with zf.open(member) as s, open(dst, "wb") as d:
        shutil.copyfileobj(s, d)

def build_train_oversampled(records):
    out_images = []
    out_annotations = []
    source_class_counts = Counter()
    effective_class_counts = Counter()
    multiplier_counts = Counter()
    next_image_id = 1
    next_ann_id = 1

    for rec in tqdm(records, desc="build oversampled train"):
        cats = {int(a["category_id"]) for a in rec["anns"]}
        mult = max(CATEGORY_MULTIPLIERS.get(c, 1) for c in cats)
        multiplier_counts[mult] += 1

        for a in rec["anns"]:
            source_class_counts[int(a["category_id"])] += 1

        for rep in range(mult):
            new_img_id = next_image_id
            next_image_id += 1

            stem = flat_name(Path(rec["file_name"]).with_suffix("").as_posix())
            suffix = Path(rec["file_name"]).suffix or ".jpg"
            out_name = f"{rec['source_split']}__orig{rec['orig_id']:06d}__rep{rep:02d}__{stem}{suffix}"
            out_path = LOCAL_DATA_DIR / "train" / out_name

            copy_zip_member(rec["zip_member"], out_path)

            out_images.append({
                "id": new_img_id,
                "file_name": out_name,
                "width": rec["width"],
                "height": rec["height"],
            })

            for a in rec["anns"]:
                cat = int(a["category_id"])
                bbox = list(a["bbox"])
                out_annotations.append({
                    "id": next_ann_id,
                    "image_id": new_img_id,
                    "category_id": cat,
                    "bbox": bbox,
                    "area": round(bbox[2] * bbox[3], 2),
                    "iscrowd": int(a.get("iscrowd", 0)),
                })
                next_ann_id += 1
                effective_class_counts[cat] += 1

    out_coco = {
        "info": {"description": "BuzzSpot RF-DETR train+valid annotated keyframes with rare-class oversampling"},
        "licenses": [],
        "categories": CATEGORIES,
        "images": out_images,
        "annotations": out_annotations,
    }
    ann_path = LOCAL_DATA_DIR / "train" / "_annotations.coco.json"
    ann_path.write_text(json.dumps(out_coco))

    print()
    print("TRAIN OVERSAMPLED")
    print("source unique images:", len(records))
    print("final train images:", len(out_images))
    print("final train boxes:", len(out_annotations))
    print("multiplier counts:", dict(multiplier_counts))
    print("source instance counts:", {CLASS_NAMES[k-1]: source_class_counts[k] for k in range(1, 5)})
    print("effective instance counts:", {CLASS_NAMES[k-1]: effective_class_counts[k] for k in range(1, 5)})
    print("train annotations:", ann_path)
    return out_coco

def build_valid_split(records):
    out_images = []
    out_annotations = []
    class_counts = Counter()
    next_ann_id = 1

    for rec in tqdm(records, desc="build valid sanity split"):
        out_name = f"valid__orig{rec['orig_id']:06d}__{flat_name(rec['file_name'])}"
        out_path = LOCAL_DATA_DIR / "valid" / out_name
        copy_zip_member(rec["zip_member"], out_path)

        out_images.append({
            "id": int(rec["orig_id"]),
            "file_name": out_name,
            "width": rec["width"],
            "height": rec["height"],
        })

        for a in rec["anns"]:
            cat = int(a["category_id"])
            bbox = list(a["bbox"])
            out_annotations.append({
                "id": next_ann_id,
                "image_id": int(rec["orig_id"]),
                "category_id": cat,
                "bbox": bbox,
                "area": round(bbox[2] * bbox[3], 2),
                "iscrowd": int(a.get("iscrowd", 0)),
            })
            next_ann_id += 1
            class_counts[cat] += 1

    out_coco = {
        "info": {"description": "BuzzSpot valid annotated keyframes, leaked sanity split only"},
        "licenses": [],
        "categories": CATEGORIES,
        "images": out_images,
        "annotations": out_annotations,
    }
    ann_path = LOCAL_DATA_DIR / "valid" / "_annotations.coco.json"
    ann_path.write_text(json.dumps(out_coco))

    print()
    print("VALID SANITY SPLIT, NOT OVERSAMPLED")
    print("valid images:", len(out_images))
    print("valid boxes:", len(out_annotations))
    print("valid named counts:", {CLASS_NAMES[k-1]: class_counts[k] for k in range(1, 5)})
    print("valid annotations:", ann_path)
    return out_coco

def build_test_split(test_coco):
    keyframe_images = [
        im for im in test_coco["images"]
        if im.get("is_keyframe") in [True, 1, "true", "True"]
    ]
    if not keyframe_images:
        raise RuntimeError("No test keyframes found. Check test_testphase.json is_keyframe flags.")

    out_images = []
    flat_to_id = {}
    image_id_to_path = {}

    for im in tqdm(keyframe_images, desc="extract test_testphase keyframes"):
        iid = int(im["id"])
        src = find_tar(f"test_testphase/{im['file_name']}") or find_tar(im["file_name"])
        assert src is not None, f"missing test image in tar: test_testphase/{im['file_name']}"

        out_name = flat_name(im["file_name"])
        dst = LOCAL_DATA_DIR / "test" / out_name
        with tf.extractfile(tar_members[src]) as s, open(dst, "wb") as d:
            shutil.copyfileobj(s, d)

        out_images.append({
            "id": iid,
            "file_name": out_name,
            "width": int(im.get("width", 1920)),
            "height": int(im.get("height", 1080)),
        })
        flat_to_id[out_name] = iid
        image_id_to_path[iid] = dst

    out_coco = {
        "info": {"description": "BuzzSpot test_testphase keyframes for RF-DETR submission"},
        "licenses": [],
        "categories": CATEGORIES,
        "images": out_images,
        "annotations": [],
    }
    ann_path = LOCAL_DATA_DIR / "test" / "_annotations.coco.json"
    ann_path.write_text(json.dumps(out_coco))

    print()
    print("TEST")
    print("test_testphase keyframes:", len(out_images))
    print("test annotations:", ann_path)
    return out_coco, flat_to_id, image_id_to_path

train_json = load_tar_json("train.json")
valid_json = load_tar_json("valid.json")
test_json = load_tar_json("test_testphase.json")

valid_records = get_annotated_records(valid_json, "valid", "valid")

if ENABLE_TRAINING:
    train_records = get_annotated_records(train_json, "train", "train")
    train_source_records = train_records + valid_records

    print()
    print("FINAL-FIT RF-DETR TRAIN SOURCE")
    print("train annotated images:", len(train_records))
    print("valid annotated images added:", len(valid_records))
    print("train+valid annotated images:", len(train_source_records))

    train_coco = build_train_oversampled(train_source_records)
else:
    print()
    print("ENABLE_TRAINING=False -> skipping expensive train+valid oversampled dataset rebuild.")
    train_coco = None

valid_coco = build_valid_split(valid_records)
test_coco, flat_to_id, test_image_id_to_path = build_test_split(test_json)

keyframe_ids = set(flat_to_id.values())

print()
print("RF-DETR dataset dir:", LOCAL_DATA_DIR)
if ENABLE_TRAINING:
    print("train annotations:", LOCAL_DATA_DIR / "train" / "_annotations.coco.json")
print("valid annotations:", LOCAL_DATA_DIR / "valid" / "_annotations.coco.json")
print("test annotations:", LOCAL_DATA_DIR / "test" / "_annotations.coco.json")


loaded current annotation: train.json from BuzzSpot_testphase/annotations/train.json
loaded current annotation: valid.json from BuzzSpot_testphase/annotations/valid.json
loaded current annotation: test_testphase.json from BuzzSpot_testphase/annotations/test_testphase.json

valid COCO image records: 5592
valid annotated image ids used: 932
valid skipped unannotated/context image records: 4660
valid boxes: 1116
valid class counts: {1: 934, 2: 30, 3: 95, 4: 57}
valid named counts: {'bee': 934, 'bumblebee': 30, 'hoverfly': 95, 'moth': 57}

ENABLE_TRAINING=False -> skipping expensive train+valid oversampled dataset rebuild.


build valid sanity split:   0%|          | 0/932 [00:00<?, ?it/s]


VALID SANITY SPLIT, NOT OVERSAMPLED
valid images: 932
valid boxes: 1116
valid named counts: {'bee': 934, 'bumblebee': 30, 'hoverfly': 95, 'moth': 57}
valid annotations: /content/buzz_rfdetr_trainval_os/valid/_annotations.coco.json


extract test_testphase keyframes:   0%|          | 0/4763 [00:00<?, ?it/s]


TEST
test_testphase keyframes: 4763
test annotations: /content/buzz_rfdetr_trainval_os/test/_annotations.coco.json

RF-DETR dataset dir: /content/buzz_rfdetr_trainval_os
valid annotations: /content/buzz_rfdetr_trainval_os/valid/_annotations.coco.json
test annotations: /content/buzz_rfdetr_trainval_os/test/_annotations.coco.json


## 5. Load the 1344 EMA source and train a new 1536 adaptation run


## 5A. Verify and resume the completed 1536 adaptation run

The notebook scans the existing 12-epoch run directory for the newest full
PyTorch Lightning checkpoint. A valid resume checkpoint must contain non-empty
optimizer and LR-scheduler states in addition to the model and loop state.

The total trainer limit is set to 18 epochs. Because the checkpoint already
completed 12 epochs, Lightning runs only epochs 13–18.

The original selected 12-epoch EMA weights are backed up before continuation.


In [23]:
import gc
import os
import shutil
from collections import Counter, defaultdict
from pathlib import Path

import torch


def _load_coco(split_dir):
    annotation_path = Path(split_dir) / "_annotations.coco.json"
    with open(annotation_path, "r") as file:
        return json.load(file)


def _write_coco_subset(source_split_dir, target_split_dir, num_images):
    """
    Build a small COCO subset using the images with the most annotations.
    This intentionally stress-tests object-count-dependent memory rather than
    selecting arbitrary easy images.
    """
    source_split_dir = Path(source_split_dir)
    target_split_dir = Path(target_split_dir)
    target_split_dir.mkdir(parents=True, exist_ok=True)

    coco = _load_coco(source_split_dir)

    annotations_by_image = defaultdict(list)
    for annotation in coco.get("annotations", []):
        annotations_by_image[int(annotation["image_id"])].append(annotation)

    ranked_images = sorted(
        coco["images"],
        key=lambda image: len(annotations_by_image[int(image["id"])]),
        reverse=True,
    )

    selected_images = ranked_images[: min(num_images, len(ranked_images))]
    selected_ids = {int(image["id"]) for image in selected_images}

    selected_annotations = [
        annotation
        for annotation in coco.get("annotations", [])
        if int(annotation["image_id"]) in selected_ids
    ]

    for image in selected_images:
        source_image = source_split_dir / image["file_name"]
        target_image = target_split_dir / image["file_name"]

        if target_image.exists() or target_image.is_symlink():
            target_image.unlink()

        # Symlinks avoid duplicating large 1920x1080 files in /content.
        os.symlink(source_image, target_image)

    subset = {
        "info": {
            "description": "RF-DETR 1344-to-1536 warm-start memory preflight subset",
        },
        "licenses": coco.get("licenses", []),
        "categories": coco["categories"],
        "images": selected_images,
        "annotations": selected_annotations,
    }

    annotation_path = target_split_dir / "_annotations.coco.json"
    annotation_path.write_text(json.dumps(subset))

    annotation_counts = [
        len(annotations_by_image[int(image["id"])])
        for image in selected_images
    ]

    print(
        f"Preflight {target_split_dir.name}: "
        f"{len(selected_images)} images, "
        f"{len(selected_annotations)} boxes, "
        f"boxes/image={annotation_counts}"
    )


def build_memory_preflight_dataset():
    if PREFLIGHT_DATA_DIR.exists():
        shutil.rmtree(PREFLIGHT_DATA_DIR)

    for split in ["train", "valid", "test"]:
        (PREFLIGHT_DATA_DIR / split).mkdir(parents=True, exist_ok=True)

    _write_coco_subset(
        LOCAL_DATA_DIR / "train",
        PREFLIGHT_DATA_DIR / "train",
        PREFLIGHT_TRAIN_IMAGES,
    )
    _write_coco_subset(
        LOCAL_DATA_DIR / "valid",
        PREFLIGHT_DATA_DIR / "valid",
        PREFLIGHT_VALID_IMAGES,
    )

    # RF-DETR does not train on test, but keeping a valid COCO test directory
    # makes dataset auto-detection unambiguous.
    test_coco = _load_coco(LOCAL_DATA_DIR / "test")
    tiny_test = {
        "info": {"description": "empty preflight test split"},
        "licenses": test_coco.get("licenses", []),
        "categories": test_coco["categories"],
        "images": [],
        "annotations": [],
    }
    (PREFLIGHT_DATA_DIR / "test" / "_annotations.coco.json").write_text(
        json.dumps(tiny_test)
    )


def run_memory_preflight():
    """
    Run a real one-epoch training/validation pass on a tiny subset.
    A successful pass validates the loaded 1344 checkpoint, four-class head, and batch-size memory at 1536.
    """
    print()
    print("=" * 80)
    print("RUNNING RF-DETR 1344 -> 1536 WARM-START MEMORY PREFLIGHT")
    print("=" * 80)

    build_memory_preflight_dataset()

    if PREFLIGHT_OUTPUT_DIR.exists():
        shutil.rmtree(PREFLIGHT_OUTPUT_DIR)
    PREFLIGHT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    preflight_model = None

    try:
        preflight_model = load_source_model_at_1536()
        preflight_model.train(
            dataset_dir=str(PREFLIGHT_DATA_DIR),
            output_dir=str(PREFLIGHT_OUTPUT_DIR),
            epochs=1,
            batch_size=BATCH_SIZE,
            grad_accum_steps=GRAD_ACCUM_STEPS,
            lr=LR,
            lr_encoder=LR_ENCODER,
            resolution=RESOLUTION,
            use_ema=USE_EMA,
            early_stopping=False,
            gradient_checkpointing=GRADIENT_CHECKPOINTING,
            checkpoint_interval=1,
            eval_interval=1,
            eval_max_dets=100,
            log_per_class_metrics=False,
            warmup_epochs=0.0,
            skip_best_epochs=0,
            device=DEVICE,
            seed=SEED,
        )

        print()
        print("MEMORY PREFLIGHT PASSED.")
        print(
            f"Full training will use batch={BATCH_SIZE}, "
            f"grad_accum={GRAD_ACCUM_STEPS}, resolution={RESOLUTION}."
        )

    except RuntimeError as error:
        message = str(error).lower()
        if "out of memory" in message or "cuda" in message and "memory" in message:
            raise RuntimeError(
                "RF-DETR 1344-to-1536 warm-start memory preflight failed with CUDA OOM. "
                "Set BATCH_SIZE=1 and GRAD_ACCUM_STEPS=16, restart the runtime "
                "if CUDA memory remains fragmented, and rerun unchanged. "
                "Do not lower RESOLUTION in this controlled experiment; if batch 1 still OOMs, "
                "this GPU cannot run the intended 1536 setup."
            ) from error
        raise

    finally:
        del preflight_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # The preflight artifacts are disposable and should not be confused
        # with the real Drive checkpoints.
        if PREFLIGHT_OUTPUT_DIR.exists():
            shutil.rmtree(PREFLIGHT_OUTPUT_DIR)
        if PREFLIGHT_DATA_DIR.exists():
            shutil.rmtree(PREFLIGHT_DATA_DIR)



def _full_resume_candidates():
    """
    Return newest-first possible full Lightning checkpoints.

    Best-model .pth files are intentionally included only as rejected candidates:
    the verifier below requires non-empty optimizer and scheduler state.
    """
    patterns = [
        "last.ckpt",
        "checkpoint_*.ckpt",
        "checkpoint.pth",
        "checkpoint_*.pth",
    ]
    candidates = []
    for pattern in patterns:
        candidates.extend(DRIVE_RUN_DIR.glob(pattern))

    # Remove duplicates and obvious pre-extension backups.
    unique = []
    seen = set()
    for path in candidates:
        resolved = str(path.resolve())
        if resolved in seen:
            continue
        seen.add(resolved)
        if "before_18ep_resume" in path.name:
            continue
        unique.append(path)

    return sorted(
        unique,
        key=lambda path: (path.stat().st_mtime, path.name),
        reverse=True,
    )


def _inspect_full_resume_checkpoint(path):
    """
    Load and validate a trusted local RF-DETR/Lightning checkpoint.

    A true resume must contain optimizer, scheduler, loop, callback, model, epoch,
    and global-step state. Stripped inference checkpoints are rejected.
    """
    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    required_keys = {
        "state_dict",
        "optimizer_states",
        "lr_schedulers",
        "loops",
        "callbacks",
        "epoch",
        "global_step",
    }
    missing = sorted(required_keys - set(checkpoint))

    optimizer_states = checkpoint.get("optimizer_states", [])
    lr_schedulers = checkpoint.get("lr_schedulers", [])

    if missing or not optimizer_states or not lr_schedulers:
        del checkpoint
        gc.collect()
        raise ValueError(
            f"Not a full true-resume checkpoint. Missing={missing}, "
            f"optimizer_states={len(optimizer_states)}, "
            f"lr_schedulers={len(lr_schedulers)}"
        )

    epoch = int(checkpoint["epoch"])
    global_step = int(checkpoint["global_step"])

    optimizer_lrs = []
    for optimizer_state in optimizer_states:
        for group in optimizer_state.get("param_groups", []):
            if "lr" in group:
                optimizer_lrs.append(float(group["lr"]))

    metadata = {
        "path": str(path),
        "epoch": epoch,
        "completed_epochs": epoch + 1,
        "global_step": global_step,
        "optimizer_lrs": optimizer_lrs,
        "num_optimizer_states": len(optimizer_states),
        "num_lr_schedulers": len(lr_schedulers),
    }

    del checkpoint
    gc.collect()
    return metadata


def get_resume_path():
    if not AUTO_RESUME:
        if REQUIRE_TRUE_RESUME:
            raise RuntimeError(
                "REQUIRE_TRUE_RESUME=True but AUTO_RESUME=False."
            )
        return None

    candidates = _full_resume_candidates()
    errors = []

    for candidate in candidates:
        try:
            metadata = _inspect_full_resume_checkpoint(candidate)

            assert metadata["completed_epochs"] >= 12, (
                "The selected checkpoint predates the completed 12-epoch run: "
                f"{metadata}"
            )
            assert EPOCHS > metadata["completed_epochs"], (
                f"EPOCHS={EPOCHS} must be the total target epoch count and must "
                f"exceed completed_epochs={metadata['completed_epochs']}."
            )

            print()
            print("=" * 80)
            print("TRUE AUTO-RESUME VERIFIED")
            print("Checkpoint:", candidate)
            print("Checkpoint epoch (zero-based):", metadata["epoch"])
            print("Completed epochs:", metadata["completed_epochs"])
            print("Global step:", metadata["global_step"])
            print("Restored optimizer LRs:", metadata["optimizer_lrs"])
            print("Target total epochs:", EPOCHS)
            print("Additional epochs to run:", EPOCHS - metadata["completed_epochs"])
            print("Optimizer, scheduler, EMA, callbacks, weights, and loop state will be restored.")
            print("=" * 80)
            return candidate

        except Exception as error:
            errors.append((str(candidate), repr(error)))
            print("Rejected resume candidate:", candidate.name)
            print("  reason:", repr(error))

    message = (
        "No valid full Lightning checkpoint was found for a true resume in:\n"
        f"{DRIVE_RUN_DIR}\n\n"
        "Expected a recent checkpoint_*.ckpt (or equivalent) containing "
        "non-empty optimizer_states and lr_schedulers.\n"
        "The notebook will not fall back to a fresh warm start.\n\n"
        "Candidates checked:\n"
        + "\n".join(f"- {path}: {error}" for path, error in errors)
    )
    raise FileNotFoundError(message)


def backup_pre_extension_best():
    """Preserve the selected 12-epoch EMA weights before continuation."""
    if PREVIOUS_SELECTED_WEIGHTS.exists():
        if not PRE_EXTENSION_SELECTED_BACKUP.exists():
            shutil.copy2(
                PREVIOUS_SELECTED_WEIGHTS,
                PRE_EXTENSION_SELECTED_BACKUP,
            )
            print(
                "Backed up 12-epoch selected weights to:",
                PRE_EXTENSION_SELECTED_BACKUP,
            )
        else:
            print(
                "12-epoch selected backup already exists:",
                PRE_EXTENSION_SELECTED_BACKUP,
            )
    else:
        print(
            "Warning: previous selected 12-epoch weights were not found at:",
            PREVIOUS_SELECTED_WEIGHTS,
        )

    run_best = DRIVE_RUN_DIR / "checkpoint_best_ema.pth"
    if run_best.exists() and not PRE_EXTENSION_RUN_BEST_BACKUP.exists():
        shutil.copy2(run_best, PRE_EXTENSION_RUN_BEST_BACKUP)
        print(
            "Backed up run best EMA checkpoint to:",
            PRE_EXTENSION_RUN_BEST_BACKUP,
        )


In [24]:
import gc
import shutil
from pathlib import Path


def get_rfdetr_class():
    if MODEL_SIZE.lower() == "small":
        from rfdetr import RFDETRSmall
        return RFDETRSmall
    elif MODEL_SIZE.lower() == "medium":
        from rfdetr import RFDETRMedium
        return RFDETRMedium
    elif MODEL_SIZE.lower() == "base":
        from rfdetr import RFDETRBase
        return RFDETRBase
    elif MODEL_SIZE.lower() == "large":
        from rfdetr import RFDETRLarge
        return RFDETRLarge
    else:
        raise ValueError(f"Unsupported MODEL_SIZE: {MODEL_SIZE}")


RFDETRClass = get_rfdetr_class()


def stage_source_checkpoint():
    """Copy the exact selected 1344 checkpoint locally and fail loudly if absent."""
    assert SOURCE_1344_WEIGHTS.exists(), (
        "Missing selected 1344 EMA checkpoint:\n"
        f"{SOURCE_1344_WEIGHTS}\n"
        "Do not substitute checkpoint_best_total or an unrelated checkpoint."
    )
    shutil.copy2(SOURCE_1344_WEIGHTS, LOCAL_SOURCE_1344_WEIGHTS)
    print("Staged 1344 source checkpoint:", LOCAL_SOURCE_1344_WEIGHTS)
    return LOCAL_SOURCE_1344_WEIGHTS


def load_source_model_at_1536():
    """
    Load the full trained four-class 1344 checkpoint while overriding only the
    model resolution. This preserves detector weights/head but does not restore
    the completed run's optimizer or scheduler.
    """
    stage_source_checkpoint()
    model = RFDETRClass.from_checkpoint(
        str(LOCAL_SOURCE_1344_WEIGHTS),
        resolution=RESOLUTION,
    )

    print("Warm-start model resolution:", model.model_config.resolution)
    print("Warm-start model num_classes:", model.model_config.num_classes)
    print("Warm-start class names:", model.class_names)

    assert model.model_config.resolution == RESOLUTION
    assert model.model.resolution == RESOLUTION
    assert model.model_config.num_classes == 4
    assert len(model.class_names) == 4

    # One prediction catches incompatible checkpoint/model construction before training.
    sample_id = next(iter(test_image_id_to_path))
    _ = model.predict(str(test_image_id_to_path[sample_id]), threshold=0.5)
    print("1344 EMA -> 1536 checkpoint smoke test passed.")
    return model


def checkpoint_candidates():
    candidates = []
    # Prefer EMA explicitly. This run has its own output directory and names.
    for name in [
        "checkpoint_best_ema.pth",
        "checkpoint_best_total.pth",
        "checkpoint_best_regular.pth",
        f"checkpoint_{EPOCHS}.pth",
        "checkpoint.pth",
    ]:
        path = DRIVE_RUN_DIR / name
        if path.exists():
            candidates.append(path)

    if DRIVE_SELECTED_WEIGHTS.exists():
        candidates.append(DRIVE_SELECTED_WEIGHTS)

    # Safety fallback only; current run best checkpoints remain preferred.
    if PRE_EXTENSION_SELECTED_BACKUP.exists():
        candidates.append(PRE_EXTENSION_SELECTED_BACKUP)

    for path in sorted(DRIVE_RUN_DIR.glob("checkpoint_*.pth")) if DRIVE_RUN_DIR.exists() else []:
        if path not in candidates:
            candidates.append(path)

    return candidates


def try_load_checkpoint(path):
    model = RFDETRClass.from_checkpoint(str(path))
    print("Smoke-test model resolution:", model.model_config.resolution)
    print("Smoke-test num_classes:", model.model_config.num_classes)
    assert model.model_config.resolution == RESOLUTION
    assert model.model.resolution == RESOLUTION
    assert model.model_config.num_classes == 4

    sample_id = next(iter(test_image_id_to_path))
    _ = model.predict(str(test_image_id_to_path[sample_id]), threshold=0.5)
    return model


def select_and_copy_checkpoint():
    candidates = checkpoint_candidates()
    assert candidates, f"No RF-DETR checkpoints found in {DRIVE_RUN_DIR}"

    errors = []
    for path in candidates:
        try:
            print("Trying checkpoint:", path)
            candidate_model = try_load_checkpoint(path)
            del candidate_model
            gc.collect()
            shutil.copy2(path, DRIVE_SELECTED_WEIGHTS)
            shutil.copy2(path, LOCAL_WEIGHTS)
            print("Selected checkpoint:", path)
            print("Backed up selected checkpoint to:", DRIVE_SELECTED_WEIGHTS)
            print("Copied local checkpoint:", LOCAL_WEIGHTS)
            return LOCAL_WEIGHTS
        except Exception as error:
            errors.append((str(path), repr(error)))
            print("Could not load for inference:", path)
            print("Error:", repr(error))

    raise RuntimeError(
        "No checkpoint candidate could be loaded. Errors:\n"
        + "\n".join(f"{path}: {error}" for path, error in errors)
    )


def restore_checkpoint_to_local():
    if DRIVE_SELECTED_WEIGHTS.exists():
        shutil.copy2(DRIVE_SELECTED_WEIGHTS, LOCAL_WEIGHTS)
        print("Restored selected warm-start checkpoint from:", DRIVE_SELECTED_WEIGHTS)
        return LOCAL_WEIGHTS
    return select_and_copy_checkpoint()


if ENABLE_TRAINING:
    print("ENABLE_TRAINING=True -> training RF-DETR Large 1344-to-1536 adaptation.")
    DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)

    resume_path = get_resume_path()
    assert resume_path is not None, "True resume checkpoint is required."

    backup_pre_extension_best()

    # A full checkpoint already proves the 1536 batch/memory path and restores
    # the complete state, so a fresh-run preflight is intentionally skipped.
    print("Skipping memory preflight for verified true resume.")

    # Lightning restores model, optimizer, scheduler, EMA, callback, and epoch
    # state from resume_path. The source constructor is only the shell required
    # before trainer.fit(..., ckpt_path=resume_path).
    model = load_source_model_at_1536()

    train_kwargs = dict(
        dataset_dir=str(LOCAL_DATA_DIR),
        output_dir=str(DRIVE_RUN_DIR),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        grad_accum_steps=GRAD_ACCUM_STEPS,
        lr=LR,
        lr_encoder=LR_ENCODER,
        resolution=RESOLUTION,
        use_ema=USE_EMA,
        early_stopping=EARLY_STOPPING,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        checkpoint_interval=CHECKPOINT_INTERVAL,
        eval_interval=EVAL_INTERVAL,
        eval_max_dets=500,
        log_per_class_metrics=True,
        warmup_epochs=WARMUP_EPOCHS,
        lr_scheduler=LR_SCHEDULER,
        lr_drop=LR_DROP,
        skip_best_epochs=SKIP_BEST_EPOCHS,
        device=DEVICE,
        seed=SEED,
    )

    # Hard-required full checkpoint resume.
    train_kwargs["resume"] = str(resume_path)

    print("Training args:", train_kwargs)
    print("Fresh optimizer/scheduler:", False)
    print("Warmup restarted:", False)
    print("EPOCHS is the total target, not the number of added epochs:", EPOCHS)
    model.train(**train_kwargs)

    select_and_copy_checkpoint()
else:
    print("ENABLE_TRAINING=False -> skipping training.")
    restore_checkpoint_to_local()


ENABLE_TRAINING=False -> skipping training.
Restored selected warm-start checkpoint from: /content/drive/MyDrive/BuzzSpot/weights/rfdetr_large_trainval_os_1344to1536_18ep_selected_for_inference.pth


## 6. Load the selected 1536-adapted checkpoint


In [25]:
import torch, gc

restore_checkpoint_to_local()
RFDETRClass = get_rfdetr_class()
model = RFDETRClass.from_checkpoint(str(LOCAL_WEIGHTS))

print("Loaded adapted RF-DETR checkpoint:", LOCAL_WEIGHTS)
print("Model resolution:", model.model_config.resolution)
print("Model num_classes:", model.model_config.num_classes)

assert model.model_config.resolution == RESOLUTION
assert model.model.resolution == RESOLUTION
assert model.model_config.num_classes == 4

gc.collect()
torch.cuda.empty_cache()


[2026-07-07 22:09:44] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-07-07 22:09:44] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


Restored selected warm-start checkpoint from: /content/drive/MyDrive/BuzzSpot/weights/rfdetr_large_trainval_os_1344to1536_18ep_selected_for_inference.pth


[2026-07-07 22:09:45] [WARNING] rf-detr - load_pretrain_weights: args.num_queries absent; inferred ckpt_num_queries=300 from tensor rows 3900 ÷ ckpt_group_detr=13.


Loaded adapted RF-DETR checkpoint: /content/rfdetr_1344to1536_18ep_selected_for_inference.pth
Model resolution: 1536
Model num_classes: 4


## 7. Prediction helpers and robust custom-class mapping

RF-DETR fine-tuned checkpoints expose custom class names in
`detections.data["class_name"]`. This notebook maps those names directly to
BuzzSpot category IDs and ignores any placeholder/background class instead of
guessing whether numeric IDs are zero- or one-based.


In [26]:
import json, zipfile, shutil, math, gc, os, tempfile
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image
import numpy as np
import torch
from collections import Counter

W_FULL, H_FULL = 1920, 1080

CLASS_NAME_TO_CATEGORY_ID = {
    "bee": 1,
    "bumblebee": 2,
    "hoverfly": 3,
    "moth": 4,
}

def normalize_class_name(value):
    if value is None:
        return None
    if isinstance(value, bytes):
        value = value.decode("utf-8", errors="replace")
    return str(value).strip().lower().replace("_", "").replace("-", "").replace(" ", "")

NORMALIZED_CLASS_NAME_TO_CATEGORY_ID = {
    normalize_class_name(name): category_id
    for name, category_id in CLASS_NAME_TO_CATEGORY_ID.items()
}

def detection_class_names(detections):
    data = getattr(detections, "data", None)
    if not isinstance(data, dict):
        return None

    names = data.get("class_name")
    if names is None:
        return None

    names = np.asarray(names, dtype=object)
    return names

def inspect_detection_schema(model, image_id_to_path, threshold=0.001, max_images=50):
    seen = Counter()

    for idx, (_, img_path) in enumerate(image_id_to_path.items()):
        if idx >= max_images:
            break

        detections = model.predict(str(img_path), threshold=threshold)
        class_ids = getattr(detections, "class_id", None)
        class_names = detection_class_names(detections)

        if class_ids is None:
            continue

        class_ids = np.asarray(class_ids)
        if class_names is None or len(class_names) != len(class_ids):
            raise RuntimeError(
                "Fine-tuned RF-DETR predictions did not include a matching "
                'detections.data["class_name"] array. Refusing to guess numeric class IDs.'
            )

        for class_id, class_name in zip(class_ids, class_names):
            seen[(int(class_id), str(class_name))] += 1

    print("RF-DETR raw class-id / class-name pairs seen in probe:")
    for pair, count in seen.most_common():
        print(f"  {pair}: {count}")

    resolved_names = {
        normalize_class_name(class_name)
        for (_, class_name) in seen
        if normalize_class_name(class_name) in NORMALIZED_CLASS_NAME_TO_CATEGORY_ID
    }
    print("Resolved BuzzSpot class names:", sorted(resolved_names))

    if not resolved_names:
        raise RuntimeError(
            "The checkpoint probe did not expose any recognized BuzzSpot class names."
        )

inspect_detection_schema(
    model,
    test_image_id_to_path,
    threshold=REGULAR_CONF,
    max_images=50,
)

def category_id_from_detection(class_name):
    normalized = normalize_class_name(class_name)
    return NORMALIZED_CLASS_NAME_TO_CATEGORY_ID.get(normalized)

def clip_xyxy(x1, y1, x2, y2, W, H):
    x1 = max(0.0, min(float(x1), W - 1))
    y1 = max(0.0, min(float(y1), H - 1))
    x2 = max(0.0, min(float(x2), W))
    y2 = max(0.0, min(float(y2), H))

    if x2 - x1 < 1 or y2 - y1 < 1:
        return None

    return x1, y1, x2, y2

def detections_to_records(
    detections,
    image_id,
    x_offset=0,
    y_offset=0,
    W=W_FULL,
    H=H_FULL,
    max_det=None,
):
    records = []

    if detections is None:
        return records

    xyxy = getattr(detections, "xyxy", None)
    conf = getattr(detections, "confidence", None)
    class_ids = getattr(detections, "class_id", None)
    class_names = detection_class_names(detections)

    if xyxy is None or conf is None or class_ids is None:
        return records

    xyxy = np.asarray(xyxy)
    conf = np.asarray(conf)
    class_ids = np.asarray(class_ids)

    if class_names is None or len(class_names) != len(class_ids):
        raise RuntimeError(
            'Missing or mismatched detections.data["class_name"]. '
            "Numeric RF-DETR class IDs are intentionally not guessed."
        )

    order = np.argsort(-conf)
    if max_det is not None:
        order = order[:max_det]

    skipped_unknown = 0

    for j in order:
        category_id = category_id_from_detection(class_names[j])

        # RF-DETR may emit a placeholder/background class at a very low threshold.
        # Ignore anything whose checkpoint class name is not one of the four BuzzSpot classes.
        if category_id is None:
            skipped_unknown += 1
            continue

        x1, y1, x2, y2 = xyxy[j].tolist()
        clipped = clip_xyxy(
            x1 + x_offset,
            y1 + y_offset,
            x2 + x_offset,
            y2 + y_offset,
            W,
            H,
        )

        if clipped is None:
            continue

        x1, y1, x2, y2 = clipped
        records.append({
            "image_id": int(image_id),
            "category_id": int(category_id),
            "bbox": [
                round(x1, 2),
                round(y1, 2),
                round(x2 - x1, 2),
                round(y2 - y1, 2),
            ],
            "score": round(float(conf[j]), 5),
        })

    return records

def predict_one_image_regular(model, img_path, image_id, threshold):
    detections = model.predict(str(img_path), threshold=threshold)
    return detections_to_records(
        detections,
        image_id=image_id,
        max_det=100,
    )

def write_submission(
    preds,
    pred_path,
    zip_path,
    drive_pred_path=None,
    drive_zip_path=None,
):
    with open(pred_path, "w") as f:
        json.dump(preds, f)

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
        archive.write(pred_path, arcname="predictions.json")

    if drive_pred_path is not None:
        shutil.copy(pred_path, drive_pred_path)

    if drive_zip_path is not None:
        shutil.copy(zip_path, drive_zip_path)

def validate_submission_file(pred_path, zip_path):
    preds = json.loads(Path(pred_path).read_text())

    with zipfile.ZipFile(zip_path, "r") as archive:
        names = archive.namelist()

    assert names == ["predictions.json"], names

    bad_image_ids = [
        pred for pred in preds
        if int(pred["image_id"]) not in keyframe_ids
    ]
    bad_categories = [
        pred for pred in preds
        if int(pred["category_id"]) not in {1, 2, 3, 4}
    ]
    bad_scores = [
        pred for pred in preds
        if not (0.0 <= float(pred["score"]) <= 1.0)
    ]

    assert not bad_image_ids, f"Predictions outside test keyframes: {bad_image_ids[:5]}"
    assert not bad_categories, f"Invalid categories: {bad_categories[:5]}"
    assert not bad_scores, f"Invalid scores: {bad_scores[:5]}"

    degenerate = []
    out_of_bounds = []

    for pred in preds:
        x, y, w, h = map(float, pred["bbox"])

        if w <= 0 or h <= 0:
            degenerate.append(pred)
            continue

        if (
            x < 0
            or y < 0
            or x + w > W_FULL + 1e-6
            or y + h > H_FULL + 1e-6
        ):
            out_of_bounds.append(pred)

    assert not degenerate, f"Degenerate boxes: {degenerate[:5]}"
    assert not out_of_bounds, f"Out-of-bounds boxes: {out_of_bounds[:5]}"

    counts = Counter(int(pred["category_id"]) for pred in preds)

    print("submission validation passed")
    print("detections:", len(preds))
    print("predicted images:", len({int(pred["image_id"]) for pred in preds}), "/", len(keyframe_ids))
    print("class counts:", {
        CLASS_NAMES[category_id - 1]: counts[category_id]
        for category_id in range(1, 5)
    })
    print("zip contents:", names)


[2026-07-07 22:09:45] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. For full GPU throughput (e.g. ~8x on T4 via FP16 Tensor Cores), call model.optimize_for_inference(dtype=torch.float16).


RF-DETR raw class-id / class-name pairs seen in probe:
  (0, 'bee'): 1801
  (3, 'moth'): 291
  (2, 'hoverfly'): 93
  (1, 'bumblebee'): 63
  (4, '__background__'): 1
Resolved BuzzSpot class names: ['bee', 'bumblebee', 'hoverfly', 'moth']


## 9. Regular full-frame inference on test_testphase keyframes

In [27]:
if ENABLE_REGULAR_INFERENCE:
    regular_preds = []
    for iid, img_path in tqdm(test_image_id_to_path.items(), total=len(test_image_id_to_path), desc="regular test"):
        regular_preds.extend(predict_one_image_regular(model, img_path, iid, REGULAR_CONF))

    write_submission(
        regular_preds,
        LOCAL_REG_PRED_PATH,
        LOCAL_REG_ZIP_OUT,
        DRIVE_REG_PRED_PATH,
        DRIVE_REG_ZIP_OUT,
    )
    validate_submission_file(LOCAL_REG_PRED_PATH, LOCAL_REG_ZIP_OUT)
    print("\nUpload regular RF-DETR zip:")
    print(DRIVE_REG_ZIP_OUT)
else:
    print("ENABLE_REGULAR_INFERENCE=False -> skipping regular inference.")


regular test:   0%|          | 0/4763 [00:00<?, ?it/s]

submission validation passed
detections: 119772
predicted images: 4758 / 4763
class counts: {'bee': 94846, 'bumblebee': 3518, 'hoverfly': 13422, 'moth': 7986}
zip contents: ['predictions.json']

Upload regular RF-DETR zip:
/content/drive/MyDrive/BuzzSpot/submissions/submission_rfdetr_large_trainval_os_1344to1536_18ep_regular_conf010.zip


In [28]:
# Analyze confidence thresholds on leaked validation predictions,
# then optionally create a filtered top-100 test submission.
#
# First run:
#     SELECTED_THRESHOLD = None
#
# After reviewing the table:
#     SELECTED_THRESHOLD = 0.05  # example only; do not choose blindly

import json
import zipfile
import shutil
from collections import defaultdict, Counter
from contextlib import redirect_stdout
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


THRESHOLDS = [
    0.001,
    0.005,
    0.01,
    0.02,
    0.03,
    0.05,
    0.075,
    0.10,
    0.15,
    0.20,
    0.30,
]

MAX_DETS_PER_IMAGE = 100
SELECTED_THRESHOLD = 0.01

def load_json_from_first_existing(*paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            print("Loading:", path)
            return json.loads(path.read_text())

    raise FileNotFoundError(
        "Could not find any of these files:\n"
        + "\n".join(str(path) for path in paths)
    )


def filter_and_cap_predictions(
    predictions,
    confidence_threshold,
    max_detections_per_image=100,
):
    grouped = defaultdict(list)

    for prediction in predictions:
        if float(prediction["score"]) >= confidence_threshold:
            grouped[int(prediction["image_id"])].append(prediction)

    filtered = []

    for image_id, image_predictions in grouped.items():
        image_predictions = sorted(
            image_predictions,
            key=lambda prediction: float(prediction["score"]),
            reverse=True,
        )

        filtered.extend(
            image_predictions[:max_detections_per_image]
        )

    return filtered


def prediction_count_stats(predictions, all_image_ids):
    counts = Counter(
        int(prediction["image_id"])
        for prediction in predictions
    )

    values = np.array(
        [counts.get(int(image_id), 0) for image_id in all_image_ids],
        dtype=np.int32,
    )

    return {
        "detections": int(len(predictions)),
        "mean_per_image": float(values.mean()),
        "median_per_image": float(np.median(values)),
        "max_per_image": int(values.max()) if len(values) else 0,
        "zero_images": int((values == 0).sum()),
    }


def evaluate_coco_predictions(coco_gt, predictions):
    # Suppress COCO's repeated console output during the sweep.
    with redirect_stdout(StringIO()):
        coco_dt = coco_gt.loadRes(predictions if predictions else [])

        evaluator = COCOeval(coco_gt, coco_dt, "bbox")
        evaluator.params.maxDets = [1, 10, 100]
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()

    precision = evaluator.eval["precision"]

    per_class_ap = {}

    for class_index, class_name in enumerate(CLASS_NAMES):
        class_precision = precision[
            :, :, class_index, 0, -1
        ]

        valid_precision = class_precision[class_precision > -1]

        per_class_ap[class_name] = (
            float(valid_precision.mean())
            if valid_precision.size
            else float("nan")
        )

    return {
        "mAP50_95": float(evaluator.stats[0]),
        "mAP50": float(evaluator.stats[1]),
        "mAP75": float(evaluator.stats[2]),
        "mAR100": float(evaluator.stats[8]),
        **{
            f"AP_{class_name}": ap
            for class_name, ap in per_class_ap.items()
        },
    }


# Load raw leaked-validation predictions generated in Section 8.
raw_val_prediction_path = Path(
    f"/content/val_predictions_{MODEL_TAG}_{REG_TAG}.json"
)

raw_val_predictions = load_json_from_first_existing(
    raw_val_prediction_path,
)

# Load raw test predictions generated in Section 9 when available.
# This allows validation-only sweeps without failing if test inference is disabled
# or still pending.
raw_test_predictions = None
if LOCAL_REG_PRED_PATH.exists() or DRIVE_REG_PRED_PATH.exists():
    raw_test_predictions = load_json_from_first_existing(
        LOCAL_REG_PRED_PATH,
        DRIVE_REG_PRED_PATH,
    )
else:
    print("Test predictions not generated yet — running validation sweep only.")

val_annotation_path = (
    LOCAL_DATA_DIR / "valid" / "_annotations.coco.json"
)

with redirect_stdout(StringIO()):
    val_coco_gt = COCO(str(val_annotation_path))

val_image_ids = sorted(val_coco_gt.getImgIds())

sweep_rows = []

for threshold in THRESHOLDS:
    filtered_val_predictions = filter_and_cap_predictions(
        raw_val_predictions,
        confidence_threshold=threshold,
        max_detections_per_image=MAX_DETS_PER_IMAGE,
    )

    metrics = evaluate_coco_predictions(
        val_coco_gt,
        filtered_val_predictions,
    )

    count_stats = prediction_count_stats(
        filtered_val_predictions,
        val_image_ids,
    )

    sweep_rows.append({
        "threshold": threshold,
        **metrics,
        **count_stats,
    })


sweep_df = pd.DataFrame(sweep_rows)

display_columns = [
    "threshold",
    "mAP50_95",
    "mAP50",
    "mAP75",
    "mAR100",
    "AP_bee",
    "AP_bumblebee",
    "AP_hoverfly",
    "AP_moth",
    "detections",
    "mean_per_image",
    "median_per_image",
    "max_per_image",
    "zero_images",
]

sweep_df = sweep_df[display_columns]

display(
    sweep_df.style.format({
        "threshold": "{:.3f}",
        "mAP50_95": "{:.4f}",
        "mAP50": "{:.4f}",
        "mAP75": "{:.4f}",
        "mAR100": "{:.4f}",
        "AP_bee": "{:.4f}",
        "AP_bumblebee": "{:.4f}",
        "AP_hoverfly": "{:.4f}",
        "AP_moth": "{:.4f}",
        "mean_per_image": "{:.2f}",
        "median_per_image": "{:.1f}",
    })
)

summary_path = (
    SUBMISSIONS_DIR
    / f"threshold_sweep_{MODEL_TAG}_{REG_TAG}_top100.csv"
)

sweep_df.to_csv(summary_path, index=False)
print("Saved threshold summary:", summary_path)


if SELECTED_THRESHOLD is None:
    print(
        "\nNo submission created yet.\n"
        "Review the table, set SELECTED_THRESHOLD to one of the "
        "tested values, and rerun this cell."
    )

elif raw_test_predictions is None:
    print(
        "\nValidation threshold sweep completed. "
        "No test submission was created because test predictions are unavailable."
    )

else:
    if SELECTED_THRESHOLD not in THRESHOLDS:
        raise ValueError(
            "SELECTED_THRESHOLD must be one of the tested THRESHOLDS."
        )

    filtered_test_predictions = filter_and_cap_predictions(
        raw_test_predictions,
        confidence_threshold=SELECTED_THRESHOLD,
        max_detections_per_image=MAX_DETS_PER_IMAGE,
    )

    test_counts = Counter(
        int(prediction["image_id"])
        for prediction in filtered_test_predictions
    )

    assert not test_counts or max(test_counts.values()) <= MAX_DETS_PER_IMAGE

    threshold_tag = (
        f"conf{int(round(SELECTED_THRESHOLD * 1000)):03d}"
        f"_top{MAX_DETS_PER_IMAGE}"
    )

    local_filtered_prediction_path = Path(
        f"/content/predictions_{MODEL_TAG}_regular_{threshold_tag}.json"
    )

    local_filtered_zip_path = Path(
        f"/content/submission_{MODEL_TAG}_regular_{threshold_tag}.zip"
    )

    drive_filtered_prediction_path = (
        SUBMISSIONS_DIR / local_filtered_prediction_path.name
    )

    drive_filtered_zip_path = (
        SUBMISSIONS_DIR / local_filtered_zip_path.name
    )

    write_submission(
        filtered_test_predictions,
        local_filtered_prediction_path,
        local_filtered_zip_path,
        drive_filtered_prediction_path,
        drive_filtered_zip_path,
    )

    validate_submission_file(
        local_filtered_prediction_path,
        local_filtered_zip_path,
    )

    test_stats = prediction_count_stats(
        filtered_test_predictions,
        keyframe_ids,
    )

    print("\nFiltered test statistics:")
    print(test_stats)

    print("\nCorrected submission zip:")
    print(drive_filtered_zip_path)

Loading: /content/val_predictions_rfdetr_large_trainval_os_1344to1536_18ep_regular_conf010.json
Loading: /content/predictions_rfdetr_large_trainval_os_1344to1536_18ep_regular_conf010.json


,threshold,mAP50_95,mAP50,mAP75,mAR100,AP_bee,AP_bumblebee,AP_hoverfly,AP_moth,detections,mean_per_image,median_per_image,max_per_image,zero_images
0,0.001,0.9275,0.9973,0.9811,0.9388,0.8997,0.9900,0.8415,0.9788,4468,4.79,4.0,60,0
1,0.005,0.9275,0.9973,0.9811,0.9388,0.8997,0.9900,0.8415,0.9788,4468,4.79,4.0,60,0
2,0.010,0.9275,0.9973,0.9811,0.9388,0.8997,0.9900,0.8415,0.9788,4468,4.79,4.0,60,0
3,0.020,0.9271,0.9973,0.9811,0.9372,0.8979,0.9900,0.8415,0.9788,1499,1.61,1.0,13,0
4,0.030,0.9269,0.9973,0.9811,0.9372,0.8971,0.9900,0.8415,0.9788,1208,1.30,1.0,7,0
5,0.050,0.9269,0.9973,0.9811,0.9372,0.8971,0.9900,0.8415,0.9788,1134,1.22,1.0,6,0
6,0.075,0.9269,0.9973,0.9811,0.9372,0.8971,0.9900,0.8415,0.9788,1124,1.21,1.0,6,0
7,0.100,0.9269,0.9973,0.9811,0.9372,0.8971,0.9900,0.8415,0.9788,1120,1.20,1.0,6,0
8,0.150,0.9250,0.9925,0.9788,0.9356,0.8971,0.9900,0.8340,0.9788,1117,1.20,1.0,6,0
9,0.200,0.9250,0.9925,0.9788,0.9356,0.8971,0.9900,0.8340,0.9788,1116,1.20,1.0,6,0


Saved threshold summary: /content/drive/MyDrive/BuzzSpot/submissions/threshold_sweep_rfdetr_large_trainval_os_1344to1536_18ep_regular_conf010_top100.csv
submission validation passed
detections: 119772
predicted images: 4758 / 4763
class counts: {'bee': 94846, 'bumblebee': 3518, 'hoverfly': 13422, 'moth': 7986}
zip contents: ['predictions.json']

Filtered test statistics:
{'detections': 119772, 'mean_per_image': 25.14633634264119, 'median_per_image': 14.0, 'max_per_image': 100, 'zero_images': 5}

Corrected submission zip:
/content/drive/MyDrive/BuzzSpot/submissions/submission_rfdetr_large_trainval_os_1344to1536_18ep_regular_conf010_top100.zip


## 10. Optional SAHI helpers

In [29]:
def tile_starts(length, tile_size, overlap):
    if length <= tile_size:
        return [0]
    step = max(1, int(tile_size * (1 - overlap)))
    starts = list(range(0, length - tile_size + 1, step))
    last = length - tile_size
    if starts[-1] != last:
        starts.append(last)
    return starts

def bbox_xywh_to_xyxy(b):
    x, y, w, h = b
    return [x, y, x + w, y + h]

def ios_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    denom = min(area_a, area_b)
    return inter / denom if denom > 0 else 0.0

def greedy_nmm_ios(records, match_threshold=0.5, max_det=None):
    merged_all = []
    for cat in [1, 2, 3, 4]:
        items = [r for r in records if int(r["category_id"]) == cat]
        items = sorted(items, key=lambda r: float(r["score"]), reverse=True)

        while items:
            head = items.pop(0)
            head_box = bbox_xywh_to_xyxy(head["bbox"])
            group = [head]
            keep = []

            for r in items:
                if ios_xyxy(head_box, bbox_xywh_to_xyxy(r["bbox"])) >= match_threshold:
                    group.append(r)
                else:
                    keep.append(r)
            items = keep

            scores = np.array([max(float(g["score"]), 1e-6) for g in group], dtype=float)
            boxes = np.array([bbox_xywh_to_xyxy(g["bbox"]) for g in group], dtype=float)
            weights = scores / scores.sum()
            merged_box = (boxes * weights[:, None]).sum(axis=0).tolist()
            score = max(float(g["score"]) for g in group)

            clipped = clip_xyxy(*merged_box, W_FULL, H_FULL)
            if clipped is None:
                continue
            x1, y1, x2, y2 = clipped
            merged_all.append({
                "image_id": int(group[0]["image_id"]),
                "category_id": cat,
                "bbox": [round(x1, 2), round(y1, 2), round(x2 - x1, 2), round(y2 - y1, 2)],
                "score": round(score, 5),
            })

    merged_all = sorted(merged_all, key=lambda r: float(r["score"]), reverse=True)
    if max_det is not None:
        merged_all = merged_all[:max_det]
    return merged_all

def predict_one_image_sahi(model, img_path, image_id, threshold=0.05):
    img = Image.open(img_path).convert("RGB")
    W, H = img.size
    assert W == W_FULL and H == H_FULL, f"Unexpected image size {W}x{H}: {img_path}"

    xs = tile_starts(W, SAHI_SLICE_SIZE, SAHI_OVERLAP)
    ys = tile_starts(H, SAHI_SLICE_SIZE, SAHI_OVERLAP)
    all_records = []
    tile_path = Path("/content/_rfdetr_sahi_tile.jpg")

    for y0 in ys:
        for x0 in xs:
            x1 = min(x0 + SAHI_SLICE_SIZE, W)
            y1 = min(y0 + SAHI_SLICE_SIZE, H)
            tile = img.crop((x0, y0, x1, y1))
            tile.save(tile_path, quality=95)

            det = model.predict(str(tile_path), threshold=threshold)
            all_records.extend(detections_to_records(det, image_id=image_id, x_offset=x0, y_offset=y0, W=W, H=H, max_det=300))

    return greedy_nmm_ios(
        all_records,
        match_threshold=SAHI_MATCH_THRESHOLD,
        max_det=SAHI_MAX_DET_PER_IMAGE,
    )


## 11. Optional SAHI inference on test_testphase keyframes

In [30]:
if ENABLE_SAHI_INFERENCE:
    sahi_preds = []
    for iid, img_path in tqdm(test_image_id_to_path.items(), total=len(test_image_id_to_path), desc="SAHI test"):
        sahi_preds.extend(predict_one_image_sahi(model, img_path, iid, SAHI_CONF))

    write_submission(
        sahi_preds,
        LOCAL_SAHI_PRED_PATH,
        LOCAL_SAHI_ZIP_OUT,
        DRIVE_SAHI_PRED_PATH,
        DRIVE_SAHI_ZIP_OUT,
    )
    validate_submission_file(LOCAL_SAHI_PRED_PATH, LOCAL_SAHI_ZIP_OUT)
    print("\nUpload SAHI RF-DETR zip:")
    print(DRIVE_SAHI_ZIP_OUT)
else:
    print("ENABLE_SAHI_INFERENCE=False -> skipping SAHI inference.")


ENABLE_SAHI_INFERENCE=False -> skipping SAHI inference.


## 12. Output summary

In [31]:
print("1344 source checkpoint:", SOURCE_1344_WEIGHTS)
print("Selected adapted checkpoint:", DRIVE_SELECTED_WEIGHTS)
print("Adaptation run directory:", DRIVE_RUN_DIR)
print("Regular submission:", DRIVE_REG_ZIP_OUT if ENABLE_REGULAR_INFERENCE else "regular inference disabled")
print("SAHI submission:", DRIVE_SAHI_ZIP_OUT if ENABLE_SAHI_INFERENCE else "SAHI inference disabled")
print()
print("Continuation mode: requires a full Lightning checkpoint and restores optimizer/scheduler/EMA state.")
print("AUTO_RESUME verifies the newest full checkpoint_*.ckpt in the existing 12-epoch run directory.")
print("Memory preflight is disabled because this is a verified continuation of the same 1536 run.")
print("Target total epochs: 18; expected additional epochs from the completed run: 6.")
print("The notebook refuses to fall back to a fresh warm start if full resume state is unavailable.")
print("Custom classes are preserved from the full four-class checkpoint.")


1344 source checkpoint: /content/drive/MyDrive/BuzzSpot/weights/rfdetr_large_trainval_os_1344_32ep_selected_for_inference.pth
Selected adapted checkpoint: /content/drive/MyDrive/BuzzSpot/weights/rfdetr_large_trainval_os_1344to1536_18ep_selected_for_inference.pth
Adaptation run directory: /content/drive/MyDrive/BuzzSpot/runs_rfdetr/buzz_rfdetr_large_trainval_os_1344to1536_12ep
Regular submission: /content/drive/MyDrive/BuzzSpot/submissions/submission_rfdetr_large_trainval_os_1344to1536_18ep_regular_conf010.zip
SAHI submission: SAHI inference disabled

Continuation mode: requires a full Lightning checkpoint and restores optimizer/scheduler/EMA state.
AUTO_RESUME verifies the newest full checkpoint_*.ckpt in the existing 12-epoch run directory.
Memory preflight is disabled because this is a verified continuation of the same 1536 run.
Target total epochs: 18; expected additional epochs from the completed run: 6.
The notebook refuses to fall back to a fresh warm start if full resume state i